# Database Queries and Analysis
This notebook contains queries for various database tables, identifies potential data issues, and provides insights for further analysis.

In [ ]:
# Import required libraries
import pandas as pd
import os
from utils import connect_to_db


In [ ]:
# ... (your existing imports and connect_to_db function here) ...

if __name__ == "__main__":
    print("🔄 Attempting to connect...")
    conn = connect_to_db()
    
    if conn:
        try:
            # 1. Create a cursor (the tool that executes commands)
            with conn.cursor() as cur:
                
                # Test A: Simple Ping (Ask for the current time)
                cur.execute("SELECT now();")
                time_result = cur.fetchone()
                print(f"✅ Connection Successful! Server Time: {time_result[0]}")
                
                # Test B: List all tables (Prove your migration worked)
                print("\n📊 Checking for tables...")
                cur.execute("""
                    SELECT table_name 
                    FROM information_schema.tables 
                    WHERE table_schema = 'mart';
                """)
                tables = cur.fetchall()
                
                if tables:
                    print("Found these tables in Supabase:")
                    for table in tables:
                        print(f" - {table[0]}")
                else:
                    print("⚠️ Connected, but no tables found in 'public' schema.")

        except Exception as e:
            print(f"❌ Query failed: {e}")
        
        finally:
            conn.close()
            print("\n🔌 Connection closed.")
    else:
        print("❌ Could not connect to database.")

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = """SELECT * FROM core.news WHERE tic = 'NVDA' AND published_at >= '2026-01-01';
            """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = """        WITH global_watermark AS (
                -- 1. Find the SINGLE latest catalyst date for this stock
                SELECT MAX(date) as last_run_date
                FROM core.catalyst_versions
                WHERE tic = 'NVDA'
                AND source_type = 'news'
            ),
            news_grouped AS (
                -- 2. Group news by month (same as before)
                SELECT
                    n.tic,
                    DATE_TRUNC('month', n.published_at)::DATE AS news_month,
                    COUNT(*) AS record_count,
                    MAX(n.published_at) AS latest_news_ts
                FROM core.news n
                WHERE n.tic = 'NVDA'
                AND n.published_at >= '2025-09-01'
                GROUP BY 1, 2
            )
            SELECT 
                ng.tic,
                EXTRACT(YEAR FROM ng.news_month)::INT AS year,
                NULL AS quarter,
                EXTRACT(MONTH FROM ng.news_month)::INT AS month,
                ng.record_count,
                sp.name, 
                sp.sector, 
                sp.industry, 
                sp.short_summary
            FROM news_grouped ng
            JOIN core.stock_profiles sp ON ng.tic = sp.tic
            CROSS JOIN global_watermark gw
            WHERE 
                -- 3. ONLY select if news is newer than the global high water mark
                -- (If no catalyst exists yet (NULL), we take everything)
                gw.last_run_date IS NULL
                OR ng.latest_news_ts > gw.last_run_date
            ORDER BY ng.news_month ASC;"""
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = """SELECT * FROM core.catalyst_versions ORDER BY date DESC LIMIT 5;
            """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
df.iloc[0]['raw_json']

In [ ]:
df

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
# display all rows
pd.set_option('display.max_rows', None)
# display columns width
pd.set_option('display.max_colwidth', None)
if conn:
    query = "SELECT * FROM core.catalyst_master WHERE tic = 'HOOD' AND mention_count > 0;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM mart.catalyst_master ;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM mart.catalyst_master WHERE as_of_date = '2026-01-03' AND tic = 'NVDA';"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM mart.earnings_transcript_analysis WHERE tic = 'COIN' AND as_of_date = '2026-01-01' ORDER BY calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)
    # print(df.shape)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM mart.catalyst_master WHERE sentiment = -1 AND as_of_date = '2025-12-30' AND mention_count > 0;"
    df_negative = pd.read_sql_query(query, conn)
    query = "SELECT * FROM mart.catalyst_master WHERE sentiment = 1 AND as_of_date = '2025-12-30' AND mention_count > 0;"
    df_positive = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    # display(df)
    print(df_negative.shape)
    print(df_positive.shape)
    print("Negative Sentiment Ratio:", df_negative.shape[0] / (df_negative.shape[0] + df_positive.shape[0]))

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM mart.catalyst_master WHERE sentiment = -1 AND as_of_date = '2026-01-01' AND mention_count > 0;"
    df_negative = pd.read_sql_query(query, conn)
    query = "SELECT * FROM mart.catalyst_master WHERE sentiment = 1 AND as_of_date = '2026-01-01' AND mention_count > 0;"
    df_positive = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    # display(df)
    print(df_negative.shape)
    print(df_positive.shape)
    print("Negative Sentiment Ratio:", df_negative.shape[0] / (df_negative.shape[0] + df_positive.shape[0]))

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_master WHERE sentiment = -1 AND mention_count > 0;"
    df_negative = pd.read_sql_query(query, conn)
    query = "SELECT * FROM core.catalyst_master WHERE sentiment = 1 AND mention_count > 0;"
    df_positive = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    # display(df)
    print(df_negative.shape)
    print(df_positive.shape)
    print("Negative Sentiment Ratio:", df_negative.shape[0] / (df_negative.shape[0] + df_positive.shape[0]))

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_master WHERE sentiment = -1;"
    df_negative = pd.read_sql_query(query, conn)
    query = "SELECT * FROM core.catalyst_master WHERE sentiment = 1;"
    df_positive = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    # display(df)
    print(df_negative.shape)
    print(df_positive.shape)
    print("Negative Sentiment Ratio:", df_negative.shape[0] / (df_negative.shape[0] + df_positive.shape[0]))

In [ ]:
df_positive

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = """
    SELECT cv.*, et.calendar_year, et.calendar_quarter
    FROM core.catalyst_versions AS cv
    JOIN core.earnings_transcripts AS et
        ON cv.event_id = et.event_id
    LIMIT 10;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)
    # print(df.shape)

In [ ]:
10/(29)

In [ ]:
(25, 16)
(70, 16)
(64, 16)
(162, 16)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM mart.catalyst_master ORDER BY tic, date DESC;"
    # display all columns 
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_columns', None)
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)
    # print(df)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_versions WHERE catalyst_id = 'a64c4c04-8548-4b9c-b90a-3a218d67db89';"
    # display all columns 
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_columns', None)
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)
    # print(df)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_transcript_analysis WHERE calendar_year = 2024;"
    # display all columns 
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_columns', None)
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)
    # print(df)

In [ ]:
# Query the `earnings_calendar` table
# display all rows
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_calendar ORDER BY tic, calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `stock_metadata` table
conn = connect_to_db()
# show all columns
pd.set_option('display.max_columns', None)
if conn:
    query = "SELECT * FROM core.revenue_metrics ORDER BY tic, calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
# display all rows
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings WHERE tic = 'TSLA' AND eps IS NOT NULL ORDER BY tic, calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings_transcripts` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_transcripts ORDER BY tic, calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings_transcript_chunks` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_transcript_chunks;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings_transcript_embeddings` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_transcript_embeddings;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings_transcript_analysis` table
conn = connect_to_db()
# display all columns
pd.set_option('display.max_columns', None)
if conn:
    query = "SELECT * FROM core.earnings_transcript_analysis;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings_transcript_analysis` table
conn = connect_to_db()
# display all columns
pd.set_option('display.max_columns', None)
if conn:
    query = "SELECT * FROM core.earnings_transcript_analysis WHERE tic = 'TSLA';"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.news_analysis;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.news_analysis WHERE tic = 'TSLA' ORDER BY updated_at desc LIMIT 5;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Display the max width columns for better readability
pd.set_option('display.max_colwidth', None)
len(df[df['impact_magnitude']==0][['tic', 'published_at', 'title', 'content','impact_magnitude', 'category', 'event_type', 'sentiment']])

In [ ]:
len(df)

In [ ]:
# Query the `news` table
conn = connect_to_db()
if conn:
    query = """
        WITH news_summary AS (
        SELECT
            n.tic,
            EXTRACT(YEAR FROM n.published_at)::INT AS year,
            EXTRACT(MONTH FROM n.published_at)::INT AS month,
            COUNT(*) AS record_count
        FROM core.news n
        GROUP BY n.tic, year, month
        )
        SELECT n.tic, n.year, n.month, n.record_count,
            sp.name, sp.sector, sp.industry, sp.short_summary
        FROM news_summary n
        JOIN core.stock_profiles AS sp
            ON n.tic = sp.tic
        ORDER BY n.tic, n.year, n.month;
        """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT tic, title, content, " \
    "       time_horizon, impact_magnitude, sentiment FROM core.news_analysis;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_master;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_master WHERE tic = 'TSLA' ORDER BY date desc, updated_at desc;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
# display max width columns for better readability
pd.set_option('display.max_colwidth', None)
if conn:
    query = "SELECT * FROM core.catalyst_versions;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
# display max width columns for better readability
pd.set_option('display.max_colwidth', None)
# display all columns
pd.set_option('display.max_columns', None)
if conn:
    query = "SELECT * FROM core.income_statements_quarterly WHERE tic = 'TSM' LIMIT 10;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
# display max width columns for better readability
pd.set_option('display.max_colwidth', None)
# display all columns
pd.set_option('display.max_columns', None)
if conn:
    query = "SELECT * FROM core.catalyst_versions WHERE catalyst_id = '3cee3639-df98-413b-a675-9d2a5690ce02' AND event_id = '03b75b38-db37-43f1-9c74-2dc181b770d5' ;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
# display max width columns for better readability
pd.set_option('display.max_colwidth', None)
if conn:
    query = """
    SELECT
        catalyst_id,
        tic,
        date,
        catalyst_type,
        title,
        summary,
        state,
        sentiment,
        time_horizon,
        impact_magnitude,
        certainty,
        impact_area,
        mention_count,
        event_ids,
        source_types,
        evidences,
        urls,
        created_at,
        updated_at
    FROM mart.catalyst_master
    WHERE as_of_date = (SELECT MAX(as_of_date) FROM mart.catalyst_master)
        AND mention_count > 0 AND (sentiment = 1 OR sentiment = -1)
        AND impact_magnitude <> -1
    ORDER BY tic, date DESC, impact_magnitude DESC, updated_at DESC, catalyst_id DESC;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_versions WHERE catalyst_id = '645adacf-eb96-4eb3-966d-c4e8c36b94e4' AND " \
    "(event_id = '03b75b38-db37-43f1-9c74-2dc181b770d5' OR event_id = 'a1e2bb8c-0ef3-45d7-b82d-64ba0b7a9c3f');"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_transcript_chunks WHERE " \
    "event_id = '03b75b38-db37-43f1-9c74-2dc181b770d5' AND (chunk_id = 8 OR chunk_id = 9);"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_versions WHERE event_id = '1ee975fb-a5f5-4fa2-8fd8-45c1052493fd' LIMIT 1;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:

# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = \
    """SELECT 
        cv.event_id,
        evidence,
        source_type,
        CASE WHEN source_type = 'earnings_transcript' 
             THEN 'earnings transcript (Q' || calendar_quarter || ' ' || calendar_year || ')'
             ELSE url
        END AS reference,
        calendar_quarter,
        calendar_year
        FROM core.catalyst_versions cv
        LEFT JOIN 
            (SELECT event_id, calendar_quarter, calendar_year 
            FROM core.earnings_transcripts
            WHERE tic = 'TSLA') et
        ON cv.event_id = et.event_id
        WHERE cv.tic = 'TSLA';
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_master WHERE tic = 'TSLA';"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:

# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = \
    """
        WITH catalyst_events AS (
            SELECT
                cv.event_id,
                cv.catalyst_id,
                cv.evidence,
                COALESCE(
                    CASE
                        WHEN cv.source_type = 'earnings_transcript'
                            THEN 'earnings transcript (Q' || et.calendar_quarter || ' ' || et.calendar_year || ')'
                        ELSE 'news article'
                    END
                ) AS source_type,
                cv.url
            FROM core.catalyst_versions AS cv
            LEFT JOIN (
                SELECT event_id, calendar_quarter, calendar_year
                FROM core.earnings_transcripts
                WHERE tic = 'AAPL'
            ) AS et
                ON cv.event_id = et.event_id
            WHERE cv.tic = 'AAPL' AND cv.is_valid = 1
        )
        SELECT
            cm.*,
            COALESCE(ev.source_types, ARRAY[]::text[])   AS source_types,
            COALESCE(ev.evidences, ARRAY[]::text[])      AS evidences,
            COALESCE(ev.urls, ARRAY[]::text[])     AS urls
        FROM core.catalyst_master AS cm
        LEFT JOIN LATERAL (
            SELECT
                array_agg(ce.source_type ORDER BY ce.event_id) AS source_types,
                array_agg(ce.evidence ORDER BY ce.event_id)    AS evidences,
                array_agg(ce.url ORDER BY ce.event_id)   AS urls
            FROM catalyst_events AS ce
            WHERE ce.catalyst_id = cm.catalyst_id AND ce.event_id::text = ANY(cm.event_ids)
        ) AS ev ON TRUE
        WHERE cm.mention_count > 0 AND cm.tic = 'AAPL' AND cm.catalyst_id = '645adacf-eb96-4eb3-966d-c4e8c36b94e4'
        ORDER BY cm.date DESC, cm.updated_at DESC
        LIMIT 2;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.catalyst_versions;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news_analysis` table
conn = connect_to_db()
if conn:
    # distinct count of event_id
    query = "SELECT catalyst_id, COUNT(DISTINCT event_id) AS distinct_event_count, COUNT(*) AS count FROM core.catalyst_versions GROUP BY catalyst_id ORDER BY count DESC, distinct_event_count DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# convert sentiment to text
# sentiment 1: positive
# sentiment 0: neutral
# sentiment -1: negative

df['sentiment'] = df['sentiment'].map({1: 'positive', 0: 'neutral', -1: 'negative'})

# convert impact_magnitude to text
# impact_magnitude 1: major
# impact_magnitude 0: moderate
# impact_magnitude -1: minor
df['impact_magnitude'] = df['impact_magnitude'].map({1: 'major', 0: 'moderate', -1: 'minor'})

# convert time_horizon to text
# time_horizon 0: short-term
# time_horizon 1: medium-term
# time_horizon 2 long-term
df['time_horizon'] = df['time_horizon'].map({0: 'short-term', 1: 'medium-term', 2: 'long-term'})

In [ ]:
# display all records in the DataFrame
# pd.set_option('display.max_rows', None)
# display all the content in each column
# pd.set_option('display.max_colwidth', None)
# display(df)


In [ ]:
# Query the `analyst_grade_mapping` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM ref.analyst_grade_mapping;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `analyst_grade_mapping` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.analyst_rating_monthly_summary;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
df[pd.to_datetime(df['start_date'])==pd.to_datetime("2023-11-04")]

In [ ]:
# Query the `earnings_analysis` table
conn = connect_to_db()

# display all columns
# pd.set_option('display.max_columns', None)

if conn:
    query = "SELECT * FROM core.earnings_analysis ORDER BY tic, calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_calendar ORDER BY tic, calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
df[(df['tic']=='AAPL') & (pd.to_datetime(df['earnings_date']) == pd.to_datetime('1999-12-22'))]

In [ ]:
df[(df['tic']=='AAPL')].iloc[-70:-50]

In [ ]:
# Query the `income_statements` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.balance_sheets ORDER BY tic, calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `balance_sheets` table
conn = connect_to_db()
# display all rows
pd.set_option('display.max_rows', None)
if conn:
    query = "SELECT * FROM core.balance_sheets ORDER BY tic, calendar_year DESC, calendar_quarter DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.analyst_rating_monthly_summary ORDER BY tic, start_date DESC, end_date DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.analyst_rating_quarterly_summary ORDER BY tic, start_date DESC, end_date DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.analyst_rating_monthly_summary ORDER BY tic, start_date DESC, end_date DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `news` table
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.news ORDER BY tic, published_at DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Function to chunk text
def tok_len(s: str) -> int:
    return len(enc.encode(s))

for index, row in df.iterrows(): 
    # count the number of words in the title+content column
    num_words = tok_len(str(row['title'])) + tok_len(str(row['content']))
    print(f"Row {index}: {num_words} words")    
    # if num_words > 200:
    #     print(f"Tic: {row['tic']}\nTitle: {row['title']}\nContent: {row['content']}\n")

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_calendar ORDER BY tic, earnings_date DESC;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    query = "SELECT * FROM core.earnings_calendar ORDER BY tic, earnings_date DESC;"
    _df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(_df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # group by tic count of records
    query = "SELECT tic, COUNT(*) AS record_count FROM core.earnings_calendar GROUP BY tic ORDER BY record_count DESC;"
    _df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(_df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:

    query = "SELECT DISTINCT tic FROM core.earnings_transcripts WHERE (calendar_year, calendar_quarter) = (2025, 1);"
    _df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(_df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:

    query = "SELECT * FROM core.growth_metrics WHERE tic = 'GME' ORDER BY date DESC;"
    _df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(_df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:

    query = "SELECT * FROM mart.earnings_transcript_analysis WHERE tic = 'GOOGL';"
    _df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(_df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        WITH q AS (
            SELECT DISTINCT ON (tic) *
            FROM core.stock_scores
            ORDER BY tic, date DESC
        )
        SELECT * FROM q ORDER BY total_score DESC;
    """
    
    _df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(_df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        WITH q AS (
            SELECT DISTINCT ON (tic) *
            FROM core.valuation_percentiles
            ORDER BY tic, date DESC
        )
        SELECT * FROM q;
    """
    _df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(_df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT * FROM core.catalyst_master WHERE tic = 'PLTR';
    """
    _df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(_df)

In [ ]:
from database.utils import insert_records

In [ ]:

import re

def fix_quotes(text) -> str:
    """Convert single-quote delimiters to double quotes while
    preserving apostrophes inside words (e.g., they're, it's).

    Also normalizes runs of repeated quotes down to a single character
    before applying the delimiter logic.
    """
    if text is None or pd.isnull(text):
        return text

    # Normalize multiple quotes of the same type to a single one
    text = re.sub(r"'+", "'", text)
    text = re.sub(r'"+', '"', text)

    def _replace(match: re.Match) -> str:
        idx = match.start()
        prev = text[idx - 1] if idx > 0 else ''
        next_char = text[idx + 1] if idx < len(text) - 1 else ''
        is_prev_word = prev.isalnum()
        is_next_word = next_char.isalnum()

        # Apostrophe inside a word (e.g., they're, it's) → keep as single quote
        if is_prev_word and is_next_word:
            return "'"

        # Otherwise treat as quote delimiter → convert to double quote
        return '"'

    return re.sub(r"'", _replace, text)




In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT * FROM core.earnings_transcript_analysis;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
df["past_summary"] = df["past_summary"].apply(fix_quotes)
df["future_summary"] = df["future_summary"].apply(fix_quotes)
df["risk_summary"] = df["risk_summary"].apply(fix_quotes)
df["mitigation_summary"] = df["mitigation_summary"].apply(fix_quotes)

In [ ]:
# conn = connect_to_db()
# df.drop(columns=['updated_at'], inplace=True)
# total_records = insert_records(conn, df, "core.earnings_transcript_analysis", keys=['tic', 'calendar_year', 'calendar_quarter'])
# print(total_records)
# conn.close()

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT * FROM core.catalyst_master;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
df['title'] = df['title'].apply(fix_quotes)
df['summary'] = df['summary'].apply(fix_quotes)



In [ ]:
# conn = connect_to_db()
# df.drop(columns=['updated_at'], inplace=True)
# total_records = insert_records(conn, df, "core.catalyst_master", keys=['catalyst_id'])
# print(total_records)
# conn.close()

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT * FROM core.stock_profiles;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
df['title'] = df['title'].apply(fix_quotes)
df['summary'] = df['summary'].apply(fix_quotes)
df['evidence'] = df['evidence'].apply(fix_quotes)
df['rejection_reason'] = df['rejection_reason'].apply(fix_quotes)

In [ ]:
# conn = connect_to_db()
# df.drop(columns=['updated_at'], inplace=True)
# total_records = insert_records(conn, df, "core.catalyst_versions", keys=['event_id', 'chunk_id', 'catalyst_id'])
# print(total_records)
# conn.close()

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = f"""
        SELECT vp.tic, vp.date::date,
               vp.pe_ttm_percentile, vp.pe_forward_percentile, vp.peg_ratio_forward_percentile, vp.p_to_fcf_ttm_percentile,
               pp.net_margin_percentile, pp.roe_percentile, pp.roic_percentile, pp.fcf_margin_percentile,
               gp.forward_revenue_growth_percentile, gp.revenue_growth_yoy_percentile, gp.revenue_cagr_3y_percentile, gp.ebitda_growth_yoy_percentile, gp.forward_eps_growth_percentile, gp.eps_growth_yoy_percentile,
               ep.asset_turnover_percentile, ep.dio_percentile, ep.dpo_percentile, ep.dso_percentile, ep.cash_conversion_cycle_percentile, ep.opex_ratio_percentile, ep.revenue_per_employee_percentile, ep.fixed_asset_turnover_percentile,
               fhp.interest_coverage_ttm_percentile, fhp.net_debt_to_ebitda_ttm_percentile, fhp.debt_to_assets_percentile, fhp.debt_to_equity_percentile, fhp.altman_z_score_percentile, fhp.current_ratio_percentile, fhp.cash_ratio_percentile
        FROM core.valuation_percentiles vp
        JOIN core.profitability_percentiles pp ON vp.tic = pp.tic AND vp.date = pp.date
        JOIN core.growth_percentiles gp ON vp.tic = gp.tic AND vp.date = gp.date
        JOIN core.efficiency_percentiles ep ON vp.tic = ep.tic AND vp.date = ep.date
        JOIN core.financial_health_percentiles fhp ON vp.tic = fhp.tic AND vp.date = fhp.date
        WHERE vp.tic = 'UBER'
        ORDER BY vp.date DESC LIMIT 10;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
import pandas as pd
import yfinance as yf
import numpy as np


def _to_float_or_nan(value) -> float:
    if value is None:
        return float("nan")
    if pd.isna(value):
        return float("nan")
    try:
        return float(value)
    except Exception:
        return float("nan")

def _nanmin(values) -> float:
    arr = np.array([_to_float_or_nan(v) for v in values], dtype=float)
    if arr.size == 0 or np.isnan(arr).all():
        return float("nan")
    return float(np.nanmin(arr))


def _nanmax(values) -> float:
    arr = np.array([_to_float_or_nan(v) for v in values], dtype=float)
    if arr.size == 0 or np.isnan(arr).all():
        return float("nan")
    return float(np.nanmax(arr))

def compute_growth_score(row):
    cols = [
        row['forward_revenue_growth_percentile'],
        row['revenue_growth_yoy_percentile'],
        row['revenue_cagr_3y_percentile'],
    ]
    if pd.notna(row.get('ebitda_growth_yoy_percentile')):
        cols.append(row['ebitda_growth_yoy_percentile'])
    if pd.notna(row.get('forward_eps_growth_percentile')):
        cols.append(row['forward_eps_growth_percentile'])
    elif pd.notna(row.get('eps_growth_yoy_percentile')):
        cols.append(row['eps_growth_yoy_percentile'])

    max_score = _nanmax(cols)
    min_score = _nanmin(cols)

    return 0.75 * max_score + 0.25 * min_score

In [ ]:
df['forward_eps_growth_percentile']

In [ ]:
df['growth_score'] = df.apply(compute_growth_score, axis=1)

In [ ]:
df['growth_score']

In [ ]:
df['description'] = df['description'].apply(fix_quotes)
df['summary'] = df['summary'].apply(fix_quotes)
df['short_summary'] = df['short_summary'].apply(fix_quotes)

In [ ]:
# conn = connect_to_db()
# df.drop(columns=['updated_at'], inplace=True)
# total_records = insert_records(conn, df, "core.stock_profiles", keys=['tic'])
# print(total_records)
# conn.close()

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT * FROM core.earnings_metrics WHERE tic = 'SOFI' ORDER BY calendar_year DESC, calendar_quarter DESC;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT tic FROM core.stock_profiles;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT * FROM mart.earnings_regime WHERE tic = 'NVDA' ORDER BY as_of_date DESC;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT * FROM core.earnings_metrics WHERE tic = 'NVDA' ORDER BY calendar_year DESC, calendar_quarter DESC LIMIT 5;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)

In [ ]:
# Query the `earnings` table
# display all records
pd.set_option('display.max_rows', None)
conn = connect_to_db()
if conn:
    # get the record of the latest date for each tic
    query = """
        SELECT * FROM core.eps_diluted_metrics WHERE tic = 'NVDA' ORDER BY calendar_year DESC, calendar_quarter DESC LIMIT 5;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Display the data
    display(df)